In [5]:
import csv
from pathlib import Path

def load_questions(csv_path="../eval/questions.csv"):
    rows = []
    with open(csv_path, mode="r", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            pages = [int(p.strip()) for p in r["expected_pages"].split(";") if p.strip()]
            phrases = [p.strip().lower() for p in r["answer_phrases"].split(";") if p.strip()]
            rows.append({
                "id": r["id"],
                "question": r["question"],
                "type": r["type"],
                "tag": r["tag"],
                "doc_id": r["doc_id"].strip(),
                "expected_pages": pages,
                "answer_phrases": phrases,
                "match": r["match"].strip(),
                "notes": r["notes"].strip(),
            })
    return rows

questions = load_questions()
print(f"Loaded {len(questions)} questions.")
for q in questions:
    print(q["id"], q["type"], q["tag"], q["expected_pages"], q["answer_phrases"])

Loaded 10 questions.
q1 in_scope table [20] ['2/100 leaves']
q2 in_scope multi [29] ['3 to 26', '6 to 15']
q3 in_scope plain [34] ['clear water']
q4 in_scope conflict [4, 7] ['25-28 dae', '18-20 das']
q5 in_scope plain [3] ['2% brine']
q6 in_scope plain [5] ['clearfield']
q7 in_scope table [5] ['120:60:40']
q8 in_scope paraphrase [56] ['solarization']
q9 out_of_scope out_of_scope [] []
q10 out_of_scope out_of_scope [] []


In [7]:
import chromadb
from chromadb.config import Settings
from langchain_ollama import OllamaEmbeddings

EMBED_MODEL = "nomic-embed-text"
EXPECTED_DIM = 768
EXPECTED_PREFIX = "none"

emb = OllamaEmbeddings(model=EMBED_MODEL)
client = chromadb.PersistentClient(
    path="../data/chroma/mitra",
    settings=Settings(anonymized_telemetry=False)
)
col = client.get_collection("mitra_chunks")

# Guard: stop immediately if collection metadata does not match query settings
meta = col.metadata or {}
assert meta.get("embed_model") == EMBED_MODEL, f"Model mismatch: {meta}"
assert meta.get("dim") == EXPECTED_DIM, f"Dim mismatch: {meta}"
assert meta.get("doc_prefix") == EXPECTED_PREFIX, f"Prefix mismatch: {meta}"

def norm_text(t):
    return " ".join(t.lower().splitlines())

def chunk_matches_phrase(chunk, q, phrase):
    if chunk["doc_id"] != q["doc_id"]:
        return False
    if chunk["page"] not in q["expected_pages"]:
        return False
    return phrase in norm_text(chunk["text"])

def eval_hits_at_k(chunks_k, q):
    if q["match"] == "any":
        return any(
            chunk_matches_phrase(c, q, ph)
            for c in chunks_k
            for ph in q["answer_phrases"]
        )
    elif q["match"] == "all":
        return all(
            any(chunk_matches_phrase(c, q, ph) for c in chunks_k)
            for ph in q["answer_phrases"]
        )
    raise ValueError(f"Unknown match rule: {q['match']}")

def first_hit_rank(chunks, q):
    for r in range(1, len(chunks) + 1):
        if eval_hits_at_k(chunks[:r], q):
            return r
    return None

eval_records = []
misses = []

for q in questions:
    qv = emb.embed_query(q["question"])
    res = col.query(query_embeddings=[qv], n_results=5)
    
    top5 = []
    for cid, dist, m, doc in zip(
        res["ids"][0], res["distances"][0], res["metadatas"][0], res["documents"][0]
    ):
        top5.append({
            "id": cid,
            "doc_id": m["doc_id"],
            "page": int(m["page"]),
            "similarity": 1.0 - float(dist),
            "text": doc,
        })
    
    if q["type"] == "in_scope":
        h1 = eval_hits_at_k(top5[:1], q)
        h3 = eval_hits_at_k(top5[:3], q)
        h5 = eval_hits_at_k(top5[:5], q)
        rank = first_hit_rank(top5, q)
        eval_records.append({
            "id": q["id"],
            "type": q["type"],
            "tag": q["tag"],
            "hit1": h1,
            "hit3": h3,
            "hit5": h5,
            "first_rank": rank,
            "top1_sim": round(top5[0]["similarity"], 4),
            "top5": top5,
        })
        if not h5:
            misses.append((q, top5[:3]))
    else:
        eval_records.append({
            "id": q["id"],
            "type": q["type"],
            "tag": q["tag"],
            "hit1": None,
            "hit3": None,
            "hit5": None,
            "first_rank": None,
            "top1_sim": round(top5[0]["similarity"], 4),
            "top5": top5,
        })

print("ID  | Tag        | Hit@1 | Hit@3 | Hit@5 | FirstRank | Top1_Sim")
print("-" * 65)
for r in eval_records:
    if r["type"] == "in_scope":
        print(f"{r['id']:<3} | {r['tag']:<10} | {str(r['hit1']):<5} | {str(r['hit3']):<5} | {str(r['hit5']):<5} | {str(r['first_rank']):<9} | {r['top1_sim']:.4f}")
    else:
        print(f"{r['id']:<3} | {r['tag']:<10} |  -    |  -    |  -    |  -        | {r['top1_sim']:.4f}")

ID  | Tag        | Hit@1 | Hit@3 | Hit@5 | FirstRank | Top1_Sim
-----------------------------------------------------------------
q1  | table      | False | False | False | None      | 0.8901
q2  | multi      | True  | True  | True  | 1         | 0.8391
q3  | plain      | True  | True  | True  | 1         | 0.8141
q4  | conflict   | False | False | False | None      | 0.7722
q5  | plain      | False | True  | True  | 3         | 0.8067
q6  | plain      | True  | True  | True  | 1         | 0.8585
q7  | table      | False | True  | True  | 2         | 0.7776
q8  | paraphrase | False | True  | True  | 3         | 0.7743
q9  | out_of_scope |  -    |  -    |  -    |  -        | 0.7557
q10 | out_of_scope |  -    |  -    |  -    |  -        | 0.6249


In [8]:
import time
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

# B2: Silent retrieve() returning a list of dicts
def retrieve(question, k=5, crop=None):
    qv = emb.embed_query(question)
    res = col.query(
        query_embeddings=[qv],
        n_results=k,
        where={"crop": crop} if crop else None,
    )
    chunks_out = []
    for cid, dist, m, doc in zip(
        res["ids"][0], res["distances"][0], res["metadatas"][0], res["documents"][0]
    ):
        chunks_out.append({
            "id": cid,
            "text": doc,
            "doc_id": m["doc_id"],
            "page": int(m["page"]),
            "similarity": round(1.0 - float(dist), 4),
        })
    return chunks_out

# B3: System prompt rules and context formatter
REFUSAL_SENTENCE = "I could not find this in my documents."

SYSTEM_PROMPT = (
    "You are an offline agricultural assistant for the MITRA rover. "
    "Follow these strict rules:\n"
    "1. Answer ONLY from the numbered context passages below.\n"
    "2. Cite the source passage numbers in square brackets like [1] or [2] after every claim.\n"
    "3. If the answer is not contained in the context, reply with EXACTLY this sentence and nothing else: "
    "'I could not find this in my documents.'\n"
    "4. If the context passages disagree or give different numbers/timings, state both versions clearly with their respective citations.\n"
    "5. Never invent or guess any number, herbicide dose, temperature, or date."
)

def format_context(chunks):
    blocks = []
    for idx, c in enumerate(chunks, start=1):
        blocks.append(f"[{idx}] ({c['doc_id']}, page {c['page']})\n{c['text'].strip()}")
    return "\n\n".join(blocks)

In [9]:
# B4: Local generator with temperature=0 for deterministic, low-hallucination output
llm = ChatOllama(model="gemma3:1b", temperature=0)

# B5: Provisional threshold from Part A score distribution (min_in=0.7722, max_out=0.7557).
# Provisional until Stage 5 expands the evaluation set.
PROVISIONAL_THRESHOLD = 0.765

def answer_question(question, k=5, crop=None):
    t0 = time.perf_counter()
    chunks = retrieve(question, k=k, crop=crop)
    t_retrieve = time.perf_counter() - t0

    top_sim = chunks[0]["similarity"] if chunks else 0.0
    sources = [
        {"ref": i + 1, "doc_id": c["doc_id"], "page": c["page"], "similarity": c["similarity"]}
        for i, c in enumerate(chunks)
    ]

    # B5: Pre-LLM threshold gate — refuse immediately without calling the LLM if retrieval is weak
    if top_sim < PROVISIONAL_THRESHOLD:
        return {
            "answer": REFUSAL_SENTENCE,
            "sources": sources,
            "top_similarity": top_sim,
            "gated": True,
            "timings": {"retrieve_s": round(t_retrieve, 3), "generate_s": 0.0},
        }

    # B4: Build messages and invoke ChatOllama
    ctx_str = format_context(chunks)
    user_prompt = f"Context:\n{ctx_str}\n\nQuestion: {question}\n\nAnswer:"

    t1 = time.perf_counter()
    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_prompt),
    ])
    t_generate = time.perf_counter() - t1

    return {
        "answer": response.content.strip(),
        "sources": sources,
        "top_similarity": top_sim,
        "gated": False,
        "timings": {"retrieve_s": round(t_retrieve, 3), "generate_s": round(t_generate, 3)},
    }

In [10]:
stage4_results = []

for q in questions:
    print(f"Running {q['id']} ({q['tag']})...")
    res = answer_question(q["question"], k=5)
    stage4_results.append((q, res))
    
    print(f"  top_sim={res['top_similarity']:.4f} | gated={res['gated']} | "
          f"retrieve={res['timings']['retrieve_s']}s | generate={res['timings']['generate_s']}s")
    print(f"  Answer: {res['answer']}\n")
    for s in res["sources"][:3]:
        print(f"    [{s['ref']}] {s['doc_id']} p.{s['page']} (sim={s['similarity']:.4f})")
    print("-" * 70)

# Save full Stage 4 outputs to ../eval/stage4_results.md for manual audit
md_lines = ["# Stage 4 Generation Results (gemma3:1b, k=5, threshold=0.765)\n"]
for q, res in stage4_results:
    md_lines.append(f"## {q['id']} ({q['type']} / {q['tag']})")
    md_lines.append(f"- **Question:** {q['question']}")
    md_lines.append(f"- **Expected:** `{q['doc_id']}` pages `{q['expected_pages']}` phrases `{q['answer_phrases']}`")
    md_lines.append(f"- **Top-1 Similarity:** `{res['top_similarity']:.4f}` (Gated: `{res['gated']}`)")
    md_lines.append(f"- **Timings:** retrieve `{res['timings']['retrieve_s']}s`, generate `{res['timings']['generate_s']}s`")
    md_lines.append(f"- **Top-5 Sources Passed to Prompt:**")
    for s in res["sources"]:
        md_lines.append(f"  - `[{s['ref']}]` `{s['doc_id']}` page `{s['page']}` (`sim={s['similarity']:.4f}`)")
    md_lines.append(f"\n**Model Answer:**\n> {res['answer']}\n")

Path("../eval/stage4_results.md").write_text("\n".join(md_lines), encoding="utf-8")
print("Saved ../eval/stage4_results.md")

Running q1 (table)...
  top_sim=0.8901 | gated=False | retrieve=0.137s | generate=32.563s
  Answer: Threshold Levels
Thresholds are the levels of pest populations at which you should take pest control action if you want to

    [1] potato_disease-pest-management_cpri_2016_manual p.19 (sim=0.8901)
    [2] potato_disease-pest-management_cpri_2016_manual p.19 (sim=0.7718)
    [3] potato_disease-pest-management_cpri_2016_manual p.68 (sim=0.7652)
----------------------------------------------------------------------
Running q2 (multi)...
  top_sim=0.8391 | gated=False | retrieve=0.164s | generate=36.611s
  Answer: The context states that indirect germination generally occurs at temperature ranges of 6 to 15 °C (optimum 12 °C) by means of zoospore production, and that direct germination takes place under warm temperature (a range of 4 to 30 °C optimum being 25 °C).  The context also specifies that >80% is essential for lesions expansion, and >90% is required for spore formation.

    [1] pot